# 🚀 TabDiff Stage 8/8: Epochs 7000 ➔ 8000
**Course Project:** Diffusion and Flow Matching Generative AI (DFMGAI)  
**Target Hardware:** Free Google Colab T4 GPU  
**Estimated Time:** ~15–20 minutes  
**Goal:** Train from Epoch 7000 to 8000 and save `model_8000.pt`.


## 1. Verify GPU Availability


In [ ]:
!nvidia-smi


## 2. Mount Google Drive (Recommended for Zero-Friction Checkpointing)
Mounting Google Drive allows all 8 notebooks to automatically pass checkpoints to each other without manual upload/download.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive connected! Checkpoints will sync automatically.")
except Exception as e:
    print("ℹ️ Running without Google Drive. You can use manual file upload below.")


## 3. Clone / Update Repository


In [ ]:
import os
if not os.path.exists("/content/Project_tabdiff"):
    !git clone https://github.com/rider9600/Project_tabdiff.git /content/Project_tabdiff
%cd /content/Project_tabdiff
!git pull origin main
%cd /content/Project_tabdiff/baseline_tabdiff


## 4. Install Dependencies & Prepare Adult Dataset


In [ ]:
!pip install -q toml tomli tomli-w category_encoders scikit-learn pandas scipy xgboost lightgbm prdc sdmetrics rdt icecream ml_collections imbalanced-learn kaleido
!python download_dataset.py
!python process_dataset.py --dataname adult


## 5. Restore Checkpoint from Stage 7 (`model_7000.pt`)
If you mounted Google Drive, it will automatically load. Otherwise, run the file uploader.


In [ ]:
import os, shutil
ckpt_dir = "tabdiff/ckpt/adult/learnable_schedule"
os.makedirs(ckpt_dir, exist_ok=True)
target_ckpt = f"{ckpt_dir}/model_7000.pt"
gdrive_dir = "/content/drive/MyDrive/TabDiff_Checkpoints"
if os.path.exists(f"{gdrive_dir}/model_7000.pt"):
    shutil.copy(f"{gdrive_dir}/model_7000.pt", target_ckpt)
    if os.path.exists(f"{gdrive_dir}/ema_model_7000.pt"):
        shutil.copy(f"{gdrive_dir}/ema_model_7000.pt", f"{ckpt_dir}/ema_model_7000.pt")
    print(f"✅ Automatically loaded model_7000.pt from Google Drive!")
elif not os.path.exists(target_ckpt):
    print("⚠️ Checkpoint not in Google Drive. Please upload model_7000.pt from your PC:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        shutil.move(fn, os.path.join(ckpt_dir, fn))
    print("✅ Upload complete!")
else:
    print(f"✅ Checkpoint already present at {target_ckpt}")


## 6. Train Epochs 7000 ➔ 8000
Runs for exactly 1,000 epochs (~15 minutes on T4 GPU) and saves `model_8000.pt`.


In [ ]:
!python main.py --dataname adult --method tabdiff --steps 8000 --check_val_every 1000 --ckpt_path tabdiff/ckpt/adult/learnable_schedule/model_7000.pt --no_wandb


## 7. Save Checkpoint `model_8000.pt` (Drive & PC Download)


In [ ]:
import os, shutil
from google.colab import files
ckpt_dir = "tabdiff/ckpt/adult/learnable_schedule"
gdrive_dir = "/content/drive/MyDrive/TabDiff_Checkpoints"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(gdrive_dir, exist_ok=True)
    if os.path.exists(f"{ckpt_dir}/model_8000.pt"):
        shutil.copy(f"{ckpt_dir}/model_8000.pt", f"{gdrive_dir}/model_8000.pt")
    if os.path.exists(f"{ckpt_dir}/ema_model_8000.pt"):
        shutil.copy(f"{ckpt_dir}/ema_model_8000.pt", f"{gdrive_dir}/ema_model_8000.pt")
    print("✅ Checkpoint backed up to Google Drive!")
zip_name = "tabdiff_stage_8_ckpt.zip"
!zip -j {zip_name} tabdiff/ckpt/adult/learnable_schedule/model_8000.pt tabdiff/ckpt/adult/learnable_schedule/ema_model_8000.pt 2>/dev/null || true
if os.path.exists(zip_name):
    files.download(zip_name)
    print(f"✅ Download prompt opened for {zip_name}")


## 8. Final Algorithmic Extension Benchmark & Results Export
Benchmark fast sampling speedup and download full paper results archive.


In [ ]:
!python ../experiments/run_benchmark.py --dataset adult --num_samples 1000
import shutil
from google.colab import files
shutil.make_archive("tabdiff_final_8000_results", "zip", "tabdiff")
files.download("tabdiff_final_8000_results.zip")
print("🎉 All 8,000 epochs complete! Final results archive downloaded.")
